# Flood-Resilient Routing Evaluation Notebook

This notebook executes the seeded evaluation protocol defined in `context/03_ROUTING_AND_EVALUATION.md` and `floodrouter/evaluate.py`.

Key properties:
- **No hard-coded edge cases**: all parameters (`region`, `n_pairs`, `min_trip_km`, `seed`, `alpha`, `beta`, `alpha_grid`, `beta_grid`, `as_of`) are loaded from `config/default.yaml`.
- **Strict temporal validation split**: edge weights $w(e) = L(e)(1 + \alpha F_{2012}(e) + \beta R(e))$ are built strictly from 2012 MODIS flood extents (`flood_2012`) and NASA POWER rainfall anomaly (`PRECTOTCORR`). Held-out validation measures route `flood_km` strictly against 2022 VIIRS maximum flood water extents (`flood_2022`). Both represent coarse satellite water detection rather than road-level inundation.
- **Connector road class comparison**: evaluates both `include_unknown_roads = True` and `include_unknown_roads = False`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from floodrouter.evaluate import run_full_evaluation

repo_root = Path.cwd()
if not (repo_root / "config" / "default.yaml").exists():
    repo_root = repo_root.parent

results = run_full_evaluation(
    config_path=repo_root / "config" / "default.yaml",
    repo_root=repo_root,
)
print("Config:", results["config"])

## 1. Headline Comparison: Baseline vs Alpha-Only (Rainfall Ablation) vs Resilient
Reported over both **All Pairs** and **Affected Pairs** (`baseline flood_km > 0`), for both **2012 In-Sample** and **2022 Held-Out** flood extents.

In [ ]:
rows = []
for method_name, m_data in results["methods"].items():
    for split_key, split_label in [
        ("in_sample_2012", "2012 In-Sample"),
        ("held_out_2022", "2022 Held-Out"),
    ]:
        for subset_key, subset_label in [
            ("all_pairs", "All Pairs"),
            ("affected_pairs", "Affected Pairs"),
        ]:
            sub = m_data[split_key][subset_key]
            rows.append({
                "Method": method_name,
                "alpha": m_data["alpha"],
                "beta": m_data["beta"],
                "Split": split_label,
                "Subset": subset_label,
                "n_pairs": sub["n_pairs"],
                "flood_km_mean": round(sub["flood_km_mean"], 4),
                "flood_km_median": round(sub["flood_km_median"], 4),
                "flood_km_p90": round(sub["flood_km_p90"], 4),
                "flood_km_total": round(sub["flood_km_total"], 3),
                "total_flood_km_reduction_pct": (
                    round(sub["total_flood_km_reduction_pct"], 2)
                    if sub["total_flood_km_reduction_pct"] is not None
                    else None
                ),
                "total_extra_distance_pct": round(sub["total_extra_distance_pct"], 2),
                "latency_median_ms": round(m_data["latency_ms"]["median"], 2),
                "latency_p95_ms": round(m_data["latency_ms"]["p95"], 2),
                "unreachable": m_data["unreachable_pairs"],
            })

summary_df = pd.DataFrame(rows)
display(summary_df)

## 2. Sensitivity Sweep (`alpha_grid x beta_grid`) and Knee Selection

In [ ]:
sweep_rows = []
for item in results["sensitivity_sweep"]:
    s12 = item["in_sample_2012"]["affected_pairs"]
    s22 = item["held_out_2022"]["affected_pairs"]
    sweep_rows.append({
        "alpha": item["alpha"],
        "beta": item["beta"],
        "2012_affected_flood_red_pct": round(s12["total_flood_km_reduction_pct"] or 0.0, 2),
        "2012_affected_extra_dist_pct": round(s12["total_extra_distance_pct"], 2),
        "2022_affected_flood_red_pct": round(s22["total_flood_km_reduction_pct"] or 0.0, 2),
        "2022_affected_extra_dist_pct": round(s22["total_extra_distance_pct"], 2),
    })

print(results["knee_selection_note"])
display(pd.DataFrame(sweep_rows))

## 3. Connector Road Class (`include_unknown_roads`) Comparison

In [ ]:
conn_rows = []
for key, val in results["connector_road_comparison"].items():
    r12 = val["resilient_vs_baseline"]["in_sample_2012"]["affected_pairs"]
    r22 = val["resilient_vs_baseline"]["held_out_2022"]["affected_pairs"]
    conn_rows.append({
        "setting": key,
        "include_unknown_roads": val["include_unknown_roads"],
        "graph_nodes": val["graph_nodes"],
        "graph_edges": val["graph_edges"],
        "total_length_km": round(val["total_length_km"], 2),
        "2012_affected_flood_red_pct": round(r12["total_flood_km_reduction_pct"] or 0.0, 2),
        "2022_affected_flood_red_pct": round(r22["total_flood_km_reduction_pct"] or 0.0, 2),
    })
display(pd.DataFrame(conn_rows))

## 4. Figures

In [ ]:
for name, fig_path in results["figures"].items():
    print(f"Figure: {name} -> {fig_path}")
    display(Image(filename=str(fig_path)))